# Week 12: Generative AI, Transformers & Diffusion
## Notebook 12: IP-Adapter: Decoupled Cross-Attention for Image Prompts

### 1. Conceptual Intuition & Theoretical Background
Conditioning text-to-image models on reference images (e.g. style transfer, subject preservation) by concatenating image and text tokens into a shared cross-attention matrix often causes text prompt details to be ignored.

**IP-Adapter (Image Prompt Adapter, Ye et al., 2023):**
Introduces **Decoupled Cross-Attention**:
- Leaves the original text cross-attention projection matrices ($W_k, W_v$) completely untouched and frozen.
- Adds dedicated parallel **Image Cross-Attention Layers** ($W_k^{\text{image}}, W_v^{\text{image}}$) trained exclusively on CLIP image embeddings $c_i$.

**Decoupled Cross-Attention Formulation:**
$$Z = \text{Softmax}\left( \frac{Q (W_k c_t)^T}{\sqrt{d}} \right) (W_v c_t) + \lambda \cdot \text{Softmax}\left( \frac{Q (W_k^{\text{image}} c_i)^T}{\sqrt{d}} \right) (W_v^{\text{image}} c_i)$$

Where $\lambda \in [0, 1]$ is a tunable inference-time blending scale factor.

---

### 2. Scratch Python Implementation: Decoupled Cross-Attention Layer

In [ ]:
import numpy as np

class DecoupledCrossAttentionScratch:
    def __init__(self, d_model=32, d_clip=16):
        self.d_model = d_model
        self.d_k = d_model
        
        # 1. Frozen Pretrained Text Cross-Attention Projections
        np.random.seed(42)
        self.W_k_text = np.random.normal(0, 0.1, (d_clip, d_model))
        self.W_v_text = np.random.normal(0, 0.1, (d_clip, d_model))
        
        # 2. Trainable Decoupled Image Cross-Attention Projections
        self.W_k_img = np.random.normal(0, 0.1, (d_clip, d_model))
        self.W_v_img = np.random.normal(0, 0.1, (d_clip, d_model))
        
    def forward(self, query_latents, text_embeds, img_embeds, lambda_scale=0.8):
        """
        query_latents: (batch, seq_q, d_model)
        text_embeds:   (batch, seq_text, d_clip)
        img_embeds:    (batch, seq_img, d_clip)
        """
        # Text Cross-Attention Pathway
        K_text = np.dot(text_embeds, self.W_k_text)
        V_text = np.dot(text_embeds, self.W_v_text)
        
        scores_text = np.matmul(query_latents, K_text.swapaxes(-2, -1)) / np.sqrt(self.d_k)
        attn_text = np.exp(scores_text - np.max(scores_text, axis=-1, keepdims=True))
        attn_text /= np.sum(attn_text, axis=-1, keepdims=True)
        out_text = np.matmul(attn_text, V_text)
        
        # Image Cross-Attention Pathway
        K_img = np.dot(img_embeds, self.W_k_img)
        V_img = np.dot(img_embeds, self.W_v_img)
        
        scores_img = np.matmul(query_latents, K_img.swapaxes(-2, -1)) / np.sqrt(self.d_k)
        attn_img = np.exp(scores_img - np.max(scores_img, axis=-1, keepdims=True))
        attn_img /= np.sum(attn_img, axis=-1, keepdims=True)
        out_img = np.matmul(attn_img, V_img)
        
        # Blended Output
        return out_text + lambda_scale * out_img

# Test Decoupled Cross-Attention
batch, seq_q, d_model, d_clip = 1, 16, 32, 16
Q = np.random.randn(batch, seq_q, d_model)
C_text = np.random.randn(batch, 5, d_clip) # 5 text tokens
C_img = np.random.randn(batch, 4, d_clip)  # 4 image tokens

ip_layer = DecoupledCrossAttentionScratch(d_model=d_model, d_clip=d_clip)

# Sweep lambda blending weight
out_text_only = ip_layer.forward(Q, C_text, C_img, lambda_scale=0.0)
out_balanced = ip_layer.forward(Q, C_text, C_img, lambda_scale=0.7)

print("--- IP-Adapter Decoupled Attention ---")
print(f"Output Shape:                          {out_balanced.shape}")
print(f"L2 Norm (Text Only - lambda=0.0):      {np.linalg.norm(out_text_only):.2f}")
print(f"L2 Norm (Balanced Text+Image - l=0.7): {np.linalg.norm(out_balanced):.2f}")
print("Image conditioning blended smoothly without modifying frozen text attention weights!")

### 3. Step-by-Step Code Walkthrough
1. **Parallel Projection Heads:** Uses separate $W_k, W_v$ projections for text and image modalities.
2. **Independent Softmax Normalization:** Prevents high-energy image tokens from washing out text attention scores.
3. **Inference Blending ($lambda$):** Allows real-time weighting of image prompt influence.

---

### 4. Industry Applications & Trade-offs
- **Virtual Try-On & Face Identity Preservation:** Conditioning diffusion generation on reference clothing or facial portraits.
- **Trade-off:** High scale weights ($\lambda > 1.2$) can cause image features to overpower text prompt specifications.

---

### 5. Top Interview Questions & Answers
**Q1: Why does IP-Adapter use decoupled cross-attention rather than concatenating text and image tokens?**
> **Answer:** Concatenating tokens forces text and image features to compete within the same softmax denominator. Because image tokens (CLIP features) carry higher magnitude and visual density, they suppress text tokens. Decoupled attention normalizes both modalities independently.

**Q2: What is the parameter footprint of IP-Adapter relative to full fine-tuning?**
> **Answer:** IP-Adapter only adds small cross-attention projection layers (typically around 22M parameters for SD 1.5), representing less than $2\%$ of the total base model weights.